In [ ]:
from pathlib import Path
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
import pandas as pd
import os
import uuid
import shutil

In [ ]:
os.environ["KEDRO_PACKAGE_NAME"] = "crispy_kedro"

workspace_dir = Path("workspace/results_v4")

tags=[
    "altrisk",
    # "reporting"
    ]


In [ ]:
# Since the notebook is in ./notebooks, set the project path to the parent directory
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    os.chdir(current_dir.parent)
    print(f"Changed directory from {current_dir} to {Path.cwd()}")
else:
    print(f"Already in correct directory: {current_dir}")

metadata = bootstrap_project(project_path=Path.cwd())

In [ ]:
assets = pd.read_csv("data/05_model_input/downloaded_assets.csv")
companies = pd.read_csv("data/05_model_input/downloaded_companies.csv")

assets_companies = pd.merge(
    assets, 
    companies, 
    on=["asset_id", "sector", "technology", "production_year"],
    how="left"
)

In [ ]:
# Prepare companies summary to select a subset of companies later down

assets_companies_filtered = assets_companies[assets_companies["ownership_type"] == "direct"]
assets_companies_filtered = assets_companies_filtered[assets_companies_filtered["production_year"] == 2025]
assets_companies_filtered["capacity_owned"] = assets_companies_filtered["capacity"] * assets_companies_filtered["ownership_percentage"]
companies_summary = assets_companies_filtered.groupby(
    ["company_id", "company_name", "sector"], as_index=False
).agg(
    capacity_owned=("capacity_owned", "sum"),
    n_assets=("asset_id", "nunique"),
    n_countries=("country_iso2", "nunique")
).assign(
    capacity_owned=lambda x: x["capacity_owned"].astype(int),
)
companies_summary=  companies_summary.sort_values(by=["n_assets", "capacity_owned"], ascending=False).reset_index(drop=True)
companies_summary

In [ ]:

workspace_dir.mkdir(parents=True, exist_ok=True)
print(f"Created workspace directory: {workspace_dir}")


In [ ]:
 
# import logging

# # quiet down Kedro loggers
# for name in [
#     "kedro",
#     "kedro.framework",
#     "kedro.runner",
#     "kedro.io",
#     "kedro.pipeline",
#     "kedro.extras",
# ]:
#     logging.getLogger(name).setLevel(logging.WARNING)

# # (optional) quiet root logger too
# logging.getLogger().setLevel(logging.WARNING)

In [ ]:
# Technology-based company sampling
import numpy as np

# Define sampling percentages for each technology (0.0 to 1.0)
tech_sampling_config = {
    'SolarCap - PV': 0.2,        # 10% sample
    'SolarCap - CSP': 1,       # 10% sample  
    'WindCap - Onshore': 0.2,    # 10% sample
    'WindCap - Offshore': 1,   # 10% sample
    'GasCap': 1,              # 5% sample
    'OilCap': 1,              # 5% sample
    'CoalCap': 1,             # 5% sample
    'GeothermalCap': 1,        # 20% sample
    'BF-EAF': 1,               # 10% sample
    'BF-BOF': 1,               # 10% sample
    'EAF': 1,                  # 10% sample
    'DRI-BOF': 1,              # 10% sample
    'DRI-EAF': 1,              # 10% sample
    'Oil': 1,                 # 5% sample
    'Gas': 1,                 # 5% sample
    'Both': 1,                 # 10% sample
    'BiomassCap': 1,           # 20% sample
    'NuclearCap': 1,           # 30% sample
    'HydroCap': 1,             # 10% sample
    'Coal': 1,                # 5% sample
    'Unknown': 1              # 5% sample
}

def sample_companies_by_technology(assets_companies_df, companies_summary_df, sampling_config, random_seed=42):
    """
    Sample companies based on technology percentages.
    
    Parameters:
    - assets_companies_df: DataFrame with asset-company relationships including technology
    - companies_summary_df: DataFrame with company summaries
    - sampling_config: Dictionary with technology -> sampling percentage
    - random_seed: Random seed for reproducibility
    
    Returns:
    - List of sampled company_ids
    """
    np.random.seed(random_seed)
    
    # Get companies by technology from assets data
    # Filter for 2025 and direct ownership like in companies_summary creation
    filtered_assets = assets_companies_df[
        (assets_companies_df["ownership_type"] == "direct") & 
        (assets_companies_df["production_year"] == 2025)
    ]
    
    # Group companies by technology
    companies_by_tech = filtered_assets.groupby('technology')['company_id'].unique().to_dict()
    
    sampled_company_ids = set()
    sampling_stats = {}
    
    for tech, companies_list in companies_by_tech.items():
        if tech in sampling_config:
            sample_pct = sampling_config[tech]
            n_companies = len(companies_list)
            n_sample = max(1, int(n_companies * sample_pct))  # At least 1 company if any exist
            
            # Sample companies for this technology
            sampled_tech_companies = np.random.choice(
                companies_list, 
                size=min(n_sample, n_companies), 
                replace=False
            )
            
            sampled_company_ids.update(sampled_tech_companies)
            sampling_stats[tech] = {
                'total_companies': n_companies,
                'sampled_companies': len(sampled_tech_companies),
                'sample_percentage': len(sampled_tech_companies) / n_companies if n_companies > 0 else 0
            }
        else:
            # If technology not in config, log it
            sampling_stats[tech] = {
                'total_companies': len(companies_list),
                'sampled_companies': 0,
                'sample_percentage': 0,
                'note': 'Technology not in sampling config'
            }
    
    # Convert to list and ensure companies exist in companies_summary
    final_company_ids = list(sampled_company_ids.intersection(set(companies_summary_df['company_id'])))
    
    # Print sampling statistics
    print("Technology Sampling Statistics:")
    print("=" * 50)
    for tech, stats in sampling_stats.items():
        if stats['sampled_companies'] > 0:
            print(f"{tech}: {stats['sampled_companies']}/{stats['total_companies']} companies ({stats['sample_percentage']:.1%})")
    
    print(f"\nTotal unique companies sampled: {len(final_company_ids)}")
    
    return final_company_ids

# Apply technology-based sampling
sampled_companies_by_tech = sample_companies_by_technology(
    assets_companies, 
    companies_summary, 
    tech_sampling_config
)

print(f"\nSample of selected company IDs: {sampled_companies_by_tech[:5]}...")


In [ ]:
# Use technology-based sampling
companies_selection = sampled_companies_by_tech

# Alternative: Use all companies (uncomment to disable sampling)
# companies_selection = None # all companies


# companies_selection = companies_summary.query(
#     "(n_assets < 100) & (n_assets > 5) & (sector == 'Power')"
#     ).company_id.tolist()
# len(companies_selection)

# to create an output to C/P in the config files to debug
# items = companies_selection
# yaml_list = "\n".join(f"- {item}" for item in items)
# print(yaml_list)

# companies_selection = None
# [
#     # multinational megacorps
#     # "CP_7876088876044165226",
#     # "CN_9186444779649860568",
#     "CN_8600108312240451561",
#     "CP_1512176126791706747",
#     # big greentech owners
#     "CN_6660639238798673502",
#     "CN_6660639238798673502",
#     "CN_5719632086864744401",
#     # big carbontech owners
#     "CN_8600108312240451561",
#     "CN_7548398708980274705",
#     "CN_5252218731344992786",
#     # random other owners, with 10-20 assets
#     "CN_8249155112555313068",
#     "CP_7671368023139165011",
#     "CN_7263620466430749129",
#     "CP_5133603177600074280",
#     "CP_1405113695717703383",
#     "CN_7237425157254272056",
#     # random other owners, with <10 assets
#     "CN_1325960156574879189",
#     "CN_8258543408338880789",
#     "CN_4206272115616897750",
#     "CN_413233497131578182",
#     "CP_2845696436723078206",
#     "CN_6870637186458717950",
#     "CN_1465642096900403277",
#     "CN_4903484062166566625",
#     "CP_4899418540238054262",
#     "CP_1564781859061095794",
# ]


# RUN

In [ ]:
# Define your parameter overrides
runs_configuration = {
    # "COFFEE_C3__company_granularity":{
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": True,
    #     "apply_retirement":False,
    #     "apply_decreasing_staggered_shock":False,
    #     "baseline_scenario": "AR6_COFFEE 1.1_CO_CurPol",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_800" # C3
        
    # },
    # "COFFEE_C3__asset_granularity_with_staggered_shock_and_retirement":{  
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": False,
    #     "apply_retirement":True,
    #     "apply_decreasing_staggered_shock":True,
    #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_800" # C3
    # },

    # "COFFEE_C2__company_granularity":{
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": True,
    #     "apply_retirement":False,
    #     "apply_decreasing_staggered_shock":False,
    #     "baseline_scenario": "AR6_COFFEE 1.1_CO_CurPol",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_400f", # C2
        
    # },
    # "COFFEE_C2__asset_granularity_with_staggered_shock_and_retirement":{  
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": False,
    #     "apply_retirement":True,
    #     "apply_decreasing_staggered_shock":True,
    #     "baseline_scenario": "AR6_COFFEE 1.1_CO_CurPol",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_400f", # C2
    # },

    # "COFFEE_C5__company_granularity":{
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": True,
    #     "apply_retirement":False,
    #     "apply_decreasing_staggered_shock":False,
    #     "baseline_scenario": "AR6_COFFEE 1.1_CO_CurPol",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_1400", # C5
        
    # },
    "COFFEE_C5__asset_granularity_with_staggered_shock_and_retirement":{  
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": False,
        "apply_retirement":True,
        "apply_decreasing_staggered_shock":True,
        "baseline_scenario": "AR6_COFFEE 1.1_CO_CurPol",
        "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_1400", # C5
    },
}




# # Define your parameter overrides
# runs_configuration = {
#     # "company_granularity_scenario_1":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": True,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario": "AR6_WITCH 5.0_EN_NPi2020_600" # C3
        
#     # },
#     # "asset_granularity_scenario_1":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario": "AR6_WITCH 5.0_EN_NPi2020_600" # C3
#     # },
#     # "asset_granularity_with_retirement_scenario_1":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":True,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario": "AR6_WITCH 5.0_EN_NPi2020_600" # C3
#     # },
#     # "asset_granularity_with_staggered_shock_scenario_1":{  
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":True,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario": "AR6_WITCH 5.0_EN_NPi2020_600" # C3
#     # },
#     # "asset_granularity_with_staggered_shock_and_retirement_scenario_1":{  
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":True,
#     #     "apply_decreasing_staggered_shock":True,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario": "AR6_WITCH 5.0_EN_NPi2020_600" # C3
#     # },
#     # "company_granularity_scenario_2":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": True,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario": "AR6_WITCH 5.0_EN_NPi2020_400f" # C1
#     # },
#     # "asset_granularity_scenario_2":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario": "AR6_WITCH 5.0_EN_NPi2020_400f" # C1
#     # },
#     # "asset_granularity_with_retirement_scenario_2":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":True,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario": "AR6_WITCH 5.0_EN_NPi2020_400f" # C1
#     # },
#     # "asset_granularity_with_staggered_shock_scenario_2":{  
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":True,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario": "AR6_WITCH 5.0_EN_NPi2020_400f" # C1
#     # },
#     # "asset_granularity_with_staggered_shock_and_retirement_scenario_2":{  
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":True,
#     #     "apply_decreasing_staggered_shock":True,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario": "AR6_WITCH 5.0_EN_NPi2020_400f" # C1
#     # },
#     # "company_granularity_scenario_3":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": True,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario":"AR6_WITCH 5.0_EN_NPi2020_1800f" #C5
#     # },
#     # "asset_granularity_scenario_3":{   
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario":"AR6_WITCH 5.0_EN_NPi2020_1800f" #C5
#     # },
#     # "asset_granularity_with_retirement_scenario_3":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":True,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario":"AR6_WITCH 5.0_EN_NPi2020_1800f" #C5
#     # },
#     # "asset_granularity_with_staggered_shock_scenario_3":{  
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":True,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario":"AR6_WITCH 5.0_EN_NPi2020_1800f" #C5
#     # },
#     # "asset_granularity_with_staggered_shock_and_retirement_scenario_3":{   
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":True,
#     #     "apply_decreasing_staggered_shock":True,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario":"AR6_WITCH 5.0_EN_NPi2020_1800f" #C5
#     # }        
# }








# # Define your parameter overrides
# runs_configuration = {
#     "company_granularity":{
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": True,
#         "apply_retirement":False,
#         "apply_decreasing_staggered_shock":False,
#     },
#     "asset_granularity":{
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": False,
#         "apply_retirement":False,
#         "apply_decreasing_staggered_shock":False,
#     },
#     "asset_granularity_with_retirement":{
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": False,
#         "apply_retirement":True,
#         "apply_decreasing_staggered_shock":False,
#     },
#     "asset_granularity_with_staggered_shock":{  
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": False,
#         "apply_retirement":False,
#         "apply_decreasing_staggered_shock":True,
#     },
#     "asset_granularity_with_staggered_shock_and_retirement":{  
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": False,
#         "apply_retirement":True,
#         "apply_decreasing_staggered_shock":True,
#     }
# }


In [ ]:
from IPython.display import clear_output
import sys
import traceback
from io import StringIO

all_late_sudden_trajectories = {}
all_staggered_shock_results = {}
all_companies_npvs = {}
all_run_params = {}

total_runs = len(runs_configuration)

for idx, (run_name, run_params) in enumerate(runs_configuration.items(), start=1):
    clear_output(wait=True)  # clears the cell output each iteration
    
    print("================================================")
    print("================================================")
    print(f"Running {run_name}...")
    print(f"Run {idx}/{total_runs}")
    print("================================================")
    print("================================================")
    
    try:
        with KedroSession.create(
            project_path=Path.cwd(),
            extra_params=run_params,
        ) as session:
            session.run(pipeline_name="__default__", tags=tags)

            run_id = uuid.uuid4()

            # late_sudden_trajectories = session.load("late_sudden_trajectories")
            late_sudden_trajectories = pd.read_csv(
                "data/07_model_output/companies_late_sudden_trajectories.csv"
            )
            late_sudden_trajectories["run_id"] = run_id
            staggered_shock_results = pd.read_csv(
                "data/07_model_output/asset_level_staggered_shock.csv"
            )
            staggered_shock_results["run_id"] = run_id
            yearly_npv_trajectories = pd.read_csv(
                "data/07_model_output/yearly_npv_trajectories.csv"
            )
            yearly_npv_trajectories["run_id"] = run_id
            asset_npvs = pd.read_csv(
                "data/07_model_output/asset_npv.csv"
            )
            asset_npvs["run_id"] = run_id
            company_technology_npvs = pd.read_csv(
                "data/07_model_output/company_technology_npv.csv"
            )
            company_technology_npvs["run_id"] = run_id
            companies_npvs = pd.read_csv(
                "data/07_model_output/company_npv.csv"
            )
            companies_npvs["run_id"] = run_id

            run_params_df = pd.DataFrame([run_params])
            run_params_df["run_id"] = run_id

            all_late_sudden_trajectories[run_name] = late_sudden_trajectories
            all_staggered_shock_results[run_name] = staggered_shock_results
            all_companies_npvs[run_name] = companies_npvs
            all_run_params[run_name] = run_params_df

            # Copy plot folders to {workspace_dir}/{run_name}/
            run_workspace_dir = workspace_dir / run_name
            run_workspace_dir.mkdir(parents=True, exist_ok=True)

            late_sudden_trajectories.to_csv(run_workspace_dir / "all_late_sudden_trajectories.csv", index=False)
            staggered_shock_results.to_csv(run_workspace_dir / "asset_level_staggered_shock.csv", index=False)
            asset_npvs.to_csv(run_workspace_dir / "asset_npv.csv", index=False)
            company_technology_npvs.to_csv(run_workspace_dir / "company_technology_npv.csv", index=False)
            companies_npvs.to_csv(run_workspace_dir / "company_npv.csv", index=False)
            yearly_npv_trajectories.to_csv(run_workspace_dir / "yearly_npv_trajectories.csv", index=False)
            run_params_df.to_csv(run_workspace_dir / "run_params.csv", index=False)
            
            if "reporting" in tags:
                # Copy companies_trajectories_plots
                src_trajectories = Path("data/08_reporting/companies_trajectories_plots")
                dst_trajectories = run_workspace_dir / "companies_trajectories_plots"
                if src_trajectories.exists():
                    if dst_trajectories.exists():
                        shutil.rmtree(dst_trajectories)
                    shutil.copytree(src_trajectories, dst_trajectories)
                    print(f"Copied companies_trajectories_plots to {dst_trajectories}")
                
                # Copy companies_staggered_shock_plots  
                src_staggered = Path("data/08_reporting/companies_staggered_shock_plots")
                dst_staggered = run_workspace_dir / "companies_staggered_shock_plots"
                if src_staggered.exists():
                    if dst_staggered.exists():
                        shutil.rmtree(dst_staggered)
                    shutil.copytree(src_staggered, dst_staggered)
                    print(f"Copied companies_staggered_shock_plots to {dst_staggered}")

                # Copy companies_staggered_shock_plots  
                src_staggered = Path("data/08_reporting/asset_financial_trajectories")
                dst_staggered = run_workspace_dir / "asset_financial_trajectories"
                if src_staggered.exists():
                    if dst_staggered.exists():
                        shutil.rmtree(dst_staggered)
                    shutil.copytree(src_staggered, dst_staggered)
                    print(f"Copied asset_financial_trajectories to {dst_staggered}")
                
        print(f"✅ Successfully completed {run_name}")
        
    except Exception as e:
        # Capture the error and traceback
        error_msg = f"❌ Error in {run_name}:\n"
        error_msg += f"Exception: {str(e)}\n"
        error_msg += f"Traceback:\n{traceback.format_exc()}\n"
        
        # Save error to file in the same root as the run folder
        error_file = workspace_dir / f"{run_name}_error.txt"
        with open(error_file, 'w') as f:
            f.write(error_msg)
        
        print(f"❌ Error in {run_name} - saved to {error_file}")
        print(f"Continuing with next run...")
        
        # Continue to next iteration
        continue


from IPython.display import clear_output

all_late_sudden_trajectories = {}
all_staggered_shock_results = {}
all_companies_npvs = {}
all_run_params = {}

total_runs = len(runs_configuration)

for idx, (run_name, run_params) in enumerate(runs_configuration.items(), start=1):
    clear_output(wait=True)  # clears the cell output each iteration
    
    print("================================================")
    print("================================================")
    print(f"Running {run_name}...")
    print(f"Run {idx}/{total_runs}")
    print("================================================")
    print("================================================")
    
    with KedroSession.create(
        project_path=Path.cwd(),
        extra_params=run_params,
    ) as session:
        session.run(pipeline_name="__default__", tags=tags)

        run_id = uuid.uuid4()

        # late_sudden_trajectories = session.load("late_sudden_trajectories")
        late_sudden_trajectories = pd.read_csv(
            "data/07_model_output/companies_late_sudden_trajectories.csv"
        )
        late_sudden_trajectories["run_id"] = run_id
        staggered_shock_results = pd.read_csv(
            "data/07_model_output/asset_level_staggered_shock.csv"
        )
        staggered_shock_results["run_id"] = run_id
        asset_npvs = pd.read_csv(
            "data/07_model_output/asset_npv.csv"
        )
        asset_npvs["run_id"] = run_id
        company_technology_npvs = pd.read_csv(
            "data/07_model_output/company_technology_npv.csv"
        )
        company_technology_npvs["run_id"] = run_id
        companies_npvs = pd.read_csv(
            "data/07_model_output/company_npv.csv"
        )
        companies_npvs["run_id"] = run_id

        run_params_df = pd.DataFrame([run_params])
        run_params_df["run_id"] = run_id

        all_late_sudden_trajectories[run_name] = late_sudden_trajectories
        all_staggered_shock_results[run_name] = staggered_shock_results
        all_companies_npvs[run_name] = companies_npvs
        all_run_params[run_name] = run_params_df

        # Copy plot folders to {workspace_dir}/{run_name}/
        run_workspace_dir = workspace_dir / run_name
        run_workspace_dir.mkdir(parents=True, exist_ok=True)

        late_sudden_trajectories.to_csv(run_workspace_dir / "all_late_sudden_trajectories.csv", index=False)
        staggered_shock_results.to_csv(run_workspace_dir / "asset_level_staggered_shock.csv", index=False)
        asset_npvs.to_csv(run_workspace_dir / "asset_npv.csv", index=False)
        company_technology_npvs.to_csv(run_workspace_dir / "company_technology_npv.csv", index=False)
        companies_npvs.to_csv(run_workspace_dir / "company_npv.csv", index=False)
        run_params_df.to_csv(run_workspace_dir / "run_params.csv", index=False)
        
        if "reporting" in tags:
            # Copy companies_trajectories_plots
            src_trajectories = Path("data/08_reporting/companies_trajectories_plots")
            dst_trajectories = run_workspace_dir / "companies_trajectories_plots"
            if src_trajectories.exists():
                if dst_trajectories.exists():
                    shutil.rmtree(dst_trajectories)
                shutil.copytree(src_trajectories, dst_trajectories)
                print(f"Copied companies_trajectories_plots to {dst_trajectories}")
            
            # Copy companies_staggered_shock_plots  
            src_staggered = Path("data/08_reporting/companies_staggered_shock_plots")
            dst_staggered = run_workspace_dir / "companies_staggered_shock_plots"
            if src_staggered.exists():
                if dst_staggered.exists():
                    shutil.rmtree(dst_staggered)
                shutil.copytree(src_staggered, dst_staggered)
                print(f"Copied companies_staggered_shock_plots to {dst_staggered}")

            # Copy companies_staggered_shock_plots  
            src_staggered = Path("data/08_reporting/asset_financial_trajectories")
            dst_staggered = run_workspace_dir / "asset_financial_trajectories"
            if src_staggered.exists():
                if dst_staggered.exists():
                    shutil.rmtree(dst_staggered)
                shutil.copytree(src_staggered, dst_staggered)
                print(f"Copied asset_financial_trajectories to {dst_staggered}")


# MULTI SCENARIO